In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
train_df = pd.read_parquet('train_final.parquet')
test_df = pd.read_parquet('test_final.parquet')

In [3]:
train_df.drop('Fwd Header Length.1',inplace=True,axis=1)
test_df.drop('Fwd Header Length.1',inplace=True,axis=1)

In [4]:
for col in train_df.columns:
    col_type = train_df[col].dtype
    if col_type != object:
        c_min = train_df[col].min()
        c_max = train_df[col].max()
        # Downcasting float64 to float32
        if str(col_type).find('float') >= 0 and c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
            train_df[col] = train_df[col].astype(np.float32)

        # Downcasting int64 to int32
        elif str(col_type).find('int') >= 0 and c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
            train_df[col] = train_df[col].astype(np.int32)

In [5]:
for col in test_df.columns:
    col_type = test_df[col].dtype
    if col_type != object:
        c_min = test_df[col].min()
        c_max = test_df[col].max()
        # Downcasting float64 to float32
        if str(col_type).find('float') >= 0 and c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
            test_df[col] = test_df[col].astype(np.float32)

        # Downcasting int64 to int32
        elif str(col_type).find('int') >= 0 and c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
            test_df[col] = test_df[col].astype(np.int32)

In [6]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1975710 entries, 0 to 1975709
Data columns (total 78 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int32  
 1   Flow Duration                int32  
 2   Total Fwd Packets            int32  
 3   Total Backward Packets       int32  
 4   Total Length of Fwd Packets  int32  
 5   Total Length of Bwd Packets  int32  
 6   Fwd Packet Length Max        int32  
 7   Fwd Packet Length Min        int32  
 8   Fwd Packet Length Mean       float32
 9   Fwd Packet Length Std        float32
 10  Bwd Packet Length Max        int32  
 11  Bwd Packet Length Min        int32  
 12  Bwd Packet Length Mean       float32
 13  Bwd Packet Length Std        float32
 14  Flow Bytes/s                 float32
 15  Flow Packets/s               float32
 16  Flow IAT Mean                float32
 17  Flow IAT Std                 float32
 18  Flow IAT Max                 int32  
 19  

In [7]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 580742 entries, 0 to 580741
Data columns (total 78 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Destination Port             580742 non-null  int32  
 1   Flow Duration                580742 non-null  int32  
 2   Total Fwd Packets            580742 non-null  int32  
 3   Total Backward Packets       580742 non-null  int32  
 4   Total Length of Fwd Packets  580742 non-null  int32  
 5   Total Length of Bwd Packets  580742 non-null  int32  
 6   Fwd Packet Length Max        580742 non-null  int32  
 7   Fwd Packet Length Min        580742 non-null  int32  
 8   Fwd Packet Length Mean       580742 non-null  float32
 9   Fwd Packet Length Std        580742 non-null  float32
 10  Bwd Packet Length Max        580742 non-null  int32  
 11  Bwd Packet Length Min        580742 non-null  int32  
 12  Bwd Packet Length Mean       580742 non-null  float32
 13 

In [8]:
train_df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Destination Port,1975710.0,8.712304e+03,1.916438e+04,0.0,53.0,80.0,443.0,65535.0
Flow Duration,1975710.0,1.748946e+07,3.620110e+07,-4.0,210.0,49172.0,5375275.5,119999998.0
Total Fwd Packets,1975710.0,1.082514e+01,8.297731e+02,1.0,2.0,2.0,6.0,219759.0
Total Backward Packets,1975710.0,1.221479e+01,1.101498e+03,0.0,1.0,2.0,5.0,291922.0
Total Length of Fwd Packets,1975710.0,5.761576e+02,1.108664e+04,0.0,18.0,70.0,352.0,12900000.0
...,...,...,...,...,...,...,...,...
Idle Mean,1975710.0,1.014440e+07,2.633460e+07,0.0,0.0,0.0,0.0,120000000.0
Idle Std,1975710.0,3.495007e+05,3.616116e+06,0.0,0.0,0.0,0.0,76900000.0
Idle Max,1975710.0,1.042352e+07,2.676999e+07,0.0,0.0,0.0,0.0,120000000.0
Idle Min,1975710.0,9.849848e+06,2.618644e+07,0.0,0.0,0.0,0.0,120000000.0


In [9]:
train_df['Attack'].value_counts()

Attack
0    1720966
4     189135
3      29999
5      25501
2       7417
6       1718
1        974
Name: count, dtype: int64

In [10]:
num_unique = train_df.nunique()
one_variable = num_unique[num_unique == 1]
not_one_variable = num_unique[num_unique > 1].index

dropped_cols = one_variable.index
train_df = train_df[not_one_variable]
test_df = test_df[not_one_variable]

print('Dropped columns:')
dropped_cols

Dropped columns:


Index(['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk',
       'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
       'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'],
      dtype='object')

In [11]:
train_df.shape

(1975710, 70)

In [13]:
from sklearn.preprocessing import StandardScaler
import joblib

features_train = train_df.drop('Attack', axis = 1)
attacks_train = train_df['Attack']
features_test = test_df.drop('Attack', axis = 1)
attacks_test = test_df['Attack']

scaler = StandardScaler()

scaled_features_train = scaler.fit_transform(features_train)
scaled_features_test = scaler.transform(features_test)

In [14]:
joblib.dump(scaler,r"scalers\standardscaler.joblib")

['scalers\\standardscaler.joblib']

In [15]:
print(f"Features Shape Train: f{features_train.shape}")
print(f"Attacks Shape Train: f{attacks_train.shape}")
print(f"Features Shape Test: f{features_test.shape}")
print(f"Attacks Shape Test: f{attacks_test.shape}")

Features Shape Train: f(1975710, 69)
Attacks Shape Train: f(1975710,)
Features Shape Test: f(580742, 69)
Attacks Shape Test: f(580742,)


In [18]:
from sklearn.decomposition import IncrementalPCA

size = len(features_train.columns) // 2
ipca = IncrementalPCA(n_components = size, batch_size = 500)
for batch in np.array_split(scaled_features_train, len(features_train) // 500):
    ipca.partial_fit(batch)

print(f'information retained: {sum(ipca.explained_variance_ratio_):.2%}')

information retained: 98.97%


In [19]:
# ipca = joblib.load(r"scalers\incrementail_pca_model.joblib")

In [20]:
transformed_features_train = ipca.transform(scaled_features_train)
transformed_features_test = ipca.transform(scaled_features_test)
new_data_train = pd.DataFrame(transformed_features_train, columns = [f'PC{i+1}' for i in range(size)])
new_data_test = pd.DataFrame(transformed_features_test, columns = [f'PC{i+1}' for i in range(size)])
new_data_train['Attack'] = attacks_train.values
new_data_test['Attack'] = attacks_test.values

In [21]:
joblib.dump(ipca,r"scalers\incrementail_pca_model.joblib")

['scalers\\incrementail_pca_model.joblib']

In [22]:
new_data_train

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,Attack
0,1.208802,0.234786,-2.655242,0.683666,-0.121411,-0.094553,-1.273503,-1.120861,0.323344,-0.186611,...,-0.039556,-0.870772,-0.137350,0.792820,-0.120318,-0.069270,-0.077878,0.063831,-0.025269,4
1,-1.989039,-0.056202,0.204666,-0.686079,0.642564,0.389450,-0.349293,0.197580,-0.147723,-0.007077,...,-0.046705,-0.315083,-0.259181,-0.783580,-0.898445,-0.034753,-0.067893,0.182043,-0.115440,4
2,-1.574430,-0.011603,-0.053697,0.301405,0.434055,1.359221,-1.779325,0.091420,0.222754,-0.075224,...,-0.447693,-0.666313,-0.242116,0.659885,-0.093238,-0.265243,0.012115,-0.136698,0.120370,4
3,-0.198642,0.237901,-1.570355,1.807515,0.172206,-0.435863,-1.289602,-0.357675,0.288467,-0.133384,...,-0.293437,-0.689351,0.224831,0.954030,-0.510553,-0.253497,0.104768,-0.224627,0.160110,4
4,-1.989242,-0.056210,0.204759,-0.686541,0.644365,0.388623,-0.348138,0.197196,-0.147151,-0.006874,...,-0.046728,-0.314960,-0.259961,-0.783403,-0.898783,-0.034943,-0.067817,0.181992,-0.115436,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975705,-1.893028,-0.060280,0.268199,-0.687264,0.625609,0.403655,-0.365445,0.208369,-0.163300,-0.006640,...,-0.061872,-0.308324,-0.250759,-0.773009,-0.862759,-0.051861,-0.055889,0.156927,-0.094355,3
1975706,7.703992,0.379179,-8.456708,-3.679983,-2.099403,-0.086719,-0.685795,-5.124013,-0.126994,-0.588425,...,0.507923,-1.329042,-0.423649,-0.507234,0.066560,-0.356979,0.211103,-0.344155,0.227779,3
1975707,10.372623,0.397627,-3.090078,-0.743395,0.917522,6.006258,6.297407,-4.362306,-2.585980,-0.256839,...,0.443499,-0.082441,-0.508542,-0.672531,-0.038634,-0.259785,0.158028,-0.011076,-0.019906,3
1975708,6.447138,-0.223346,-0.765794,-2.371417,0.609897,1.118669,1.214086,-0.676034,-0.516295,0.097103,...,-0.671532,1.144031,0.910587,0.257187,-1.345644,0.405992,0.045082,-0.036363,0.048831,3


In [23]:
new_data_test

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,Attack
0,0.396532,-0.016764,0.736349,1.654251,0.287301,1.398947,-1.625579,0.762095,0.829726,0.001612,...,-0.348643,0.051472,0.197045,-0.352285,0.819116,-0.399501,0.063066,-0.209173,0.157888,1
1,-2.205149,-0.072442,0.336514,-1.112137,0.983787,0.553186,-0.809378,0.928295,-1.050081,-0.192730,...,-0.035852,0.024614,-1.393375,0.497188,-0.745423,-0.277918,0.032452,0.030859,-0.051113,1
2,-1.606209,0.069591,-0.582883,0.733004,0.294969,0.661448,-1.308324,-0.134445,0.428145,-0.042611,...,-0.376428,-0.701140,0.256263,0.919610,0.245813,-0.088449,-0.094466,0.010953,0.026431,1
3,0.205208,0.512497,-2.800189,5.444278,1.063315,-3.244675,-1.796728,-0.024924,-0.375139,-0.350993,...,-0.107866,-0.148162,-0.324698,0.920478,0.313344,0.259164,-0.113511,0.220005,-0.120325,1
4,-2.225342,-0.071960,0.350683,-1.100207,0.899057,0.403997,-0.615672,0.897614,-1.022706,-0.172938,...,-0.008587,0.036439,-1.602765,0.498189,-0.912150,-0.252566,0.053514,0.012274,-0.038091,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
580737,-0.820353,0.238460,-1.026628,1.603303,-0.024363,0.460211,-1.664161,-0.130976,0.446495,-0.117648,...,-0.391652,-0.079806,-0.191300,-0.121524,0.593175,-0.186580,-0.093332,0.107418,-0.074713,2
580738,-0.788897,0.237154,-1.006463,1.619160,-0.019191,0.471900,-1.665104,-0.107092,0.457879,-0.115359,...,-0.389373,-0.073042,-0.189567,-0.131001,0.589885,-0.190679,-0.089498,0.101992,-0.071292,2
580739,-0.840388,0.236054,-1.009203,1.548680,-0.029151,0.512478,-1.666068,-0.132836,0.457758,-0.114574,...,-0.385450,-0.077438,-0.201581,-0.123759,0.609066,-0.158202,-0.112903,0.134937,-0.090881,2
580740,-2.109327,-0.059152,0.700205,-0.972366,3.397734,-0.861290,1.312915,-1.234437,0.894928,0.158974,...,-0.251555,-0.139735,-0.709327,-0.074933,-0.511820,-0.028229,-0.022857,0.068889,-0.046913,2


In [24]:
new_data_train.to_parquet(r'final\new_data_train.parquet')
new_data_test.to_parquet(r'final\new_data_test.parquet')

Balanced Dataset for binary Clasification

In [25]:
new_data_train['Attack'].value_counts()

Attack
0    1720966
4     189135
3      29999
5      25501
2       7417
6       1718
1        974
Name: count, dtype: int64

In [26]:
normal_traffic_train = new_data_train.loc[new_data_train['Attack'] == 0]
intrusions_train = new_data_train.loc[new_data_train['Attack'] != 0]

In [27]:
# Run these two lines to see the counts:
print(f"Normal Train Count: {len(normal_traffic_train)}")
print(f"Intrusions Train Count: {len(intrusions_train)}")

Normal Train Count: 1720966
Intrusions Train Count: 254744


In [28]:
normal_traffic_train = normal_traffic_train.sample(n = len(intrusions_train), replace = False, random_state=42)
ids_data_train = pd.concat([intrusions_train, normal_traffic_train])
ids_data_train['Attack'] = np.where((ids_data_train['Attack'] == 0), 0, 1)
bc_data_train = ids_data_train.sample(n = 30000, random_state=42)

In [29]:
# 6a. Print class distribution for the final training subset
print("Distribution for Training Subset (bc_data_train):")
print(bc_data_train['Attack'].value_counts())

Distribution for Training Subset (bc_data_train):
Attack
1    15050
0    14950
Name: count, dtype: int64


In [30]:

normal_traffic_test = new_data_test.loc[new_data_test['Attack'] == 0]
intrusions_test = new_data_test.loc[new_data_test['Attack'] != 0]

print("-" * 30)
normal_traffic_test = normal_traffic_test.sample(n = len(intrusions_test), replace = False, random_state=42)

ids_data_test = pd.concat([intrusions_test, normal_traffic_test])
ids_data_test['Attack'] = np.where((ids_data_test['Attack'] == 0), 0, 1)


bc_data_test = ids_data_test.sample(n=15000,random_state=42) 

print("Distribution for Balanced Test Set (bc_data_test):")
print(bc_data_test['Attack'].value_counts())

------------------------------
Distribution for Balanced Test Set (bc_data_test):
Attack
1    7627
0    7373
Name: count, dtype: int64


In [31]:
bc_data_train.to_parquet(r'final\train_bc.parquet')
bc_data_test.to_parquet(r'final\test_bc.parquet')

In [32]:
bc_data_train

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,Attack
1329324,-2.241215,-0.061802,0.264631,-1.213767,2.803885,-0.224636,0.338814,0.073483,0.110507,0.112163,...,0.189909,-0.097773,0.307899,-0.632106,-0.257305,-0.047878,-0.048214,0.112353,-0.073831,0
74248,-1.521128,-0.095889,0.249632,-0.644577,-0.526219,0.486386,-0.041000,1.926165,0.553710,0.182361,...,-1.860810,2.721752,-0.451639,0.030322,0.407755,-0.339697,0.033064,0.098368,-0.128476,1
1308556,-2.067909,-0.037118,0.240491,-0.113086,-1.681812,-0.494209,0.785609,0.084570,0.052075,0.063343,...,0.052080,-0.029073,-0.069178,-0.103265,0.046427,0.109285,0.034813,-0.048717,0.030643,0
102277,11.402089,-0.517518,1.361325,-0.962172,-0.567926,-0.257477,0.956538,4.540862,1.145267,0.434943,...,-0.304542,0.652023,0.492235,0.283983,-0.701109,0.281153,0.005249,0.001025,0.027621,1
577692,-2.369409,-0.074792,0.378981,-1.360113,1.880149,0.383993,-0.854617,1.267320,-1.406766,-0.238850,...,0.125519,0.030748,0.033934,0.612725,0.251119,0.032190,-0.019754,-0.009974,0.011965,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1200825,1.562895,0.007131,1.155807,2.215399,0.693604,1.966999,-1.321908,1.152407,1.191678,0.084718,...,-0.444098,0.475940,-0.108653,0.126050,0.235667,-0.383991,0.181557,-0.271652,0.173192,0
103884,-1.983801,-0.057198,0.192981,-0.733954,0.767833,0.494710,-0.443620,0.218210,-0.133449,-0.008790,...,0.013957,-0.302957,-0.254714,-0.783272,-0.912598,-0.122099,-0.082581,0.211805,-0.142782,1
205444,-2.351116,-0.063762,0.297650,-1.414347,3.620492,-0.499688,0.605980,0.070046,0.142973,0.147053,...,0.267402,-0.031099,0.745287,-0.552027,0.143478,0.027463,-0.047603,0.073055,-0.040125,0
121728,13.238756,-0.591768,1.508280,-1.216163,-0.180263,0.088740,1.041661,5.125994,1.580010,0.533666,...,0.683102,-0.268744,0.079301,0.469645,-0.580664,0.494552,-0.223245,0.382999,-0.273090,1


In [33]:
bc_data_test

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC26,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,Attack
364485,-1.825666,-0.063411,0.814334,-0.848239,3.066240,-1.137909,1.550857,-1.289187,0.848901,0.169473,...,0.065525,0.050259,-0.581328,0.111362,-0.425686,-0.235040,-0.007771,0.036293,-0.020672,0
574773,13.609039,-0.306607,-3.391761,-4.748557,-0.003449,-0.609088,0.622033,-2.232151,-1.157722,-0.205245,...,0.373426,-0.754912,-1.514526,-0.648662,1.934563,-0.559279,-0.058223,0.055754,-0.086467,1
505390,-2.181956,-0.072192,0.354911,-1.031508,0.712308,0.462112,-0.691707,0.834434,-1.009302,-0.183003,...,-0.060250,0.031645,-1.657954,0.487149,-0.927590,-0.277368,0.050807,0.020912,-0.046283,0
259608,-2.004710,-0.037941,0.204876,-0.145746,-1.902204,-0.482024,0.831457,0.026008,0.065910,0.063531,...,0.478866,0.145139,0.055590,0.044007,0.038733,-0.168223,-0.014137,0.047515,-0.048189,0
134534,-1.287806,-0.115100,0.621436,-0.776616,0.669954,0.399983,-0.310425,0.289829,-0.220843,0.011850,...,-0.139312,-0.298406,-0.239230,-0.680711,-0.707448,-0.108335,-0.036054,0.091893,-0.042002,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363415,-2.002690,-0.034444,0.195544,-0.133869,-1.052531,-0.143620,0.386944,-0.027318,0.206854,0.071574,...,0.380715,-0.149563,-0.108127,0.331580,-0.201626,0.168775,0.038064,-0.043793,0.020431,0
49012,-2.009943,0.025738,-0.352884,0.257244,0.315204,1.158198,-1.481087,-0.029901,0.394968,-0.056589,...,-0.333957,-0.309376,0.879379,0.068261,1.186072,-0.232620,-0.066283,-0.082895,0.083287,1
294812,-1.904943,0.002534,-0.236138,-0.056238,0.959732,0.974124,-1.457558,-0.464139,0.255458,-0.072550,...,1.012647,0.806836,-0.317593,-2.017280,-0.204841,-0.185075,0.150758,-0.085998,0.012479,0
254155,0.235701,0.143622,-1.698717,0.397528,0.050431,1.187976,-1.282234,-1.169468,0.467234,-0.111412,...,0.062896,-0.237319,0.113913,0.598329,0.083925,-0.007891,-0.058496,0.018420,0.000311,0


BAlANCED DATA FOR MULTICLASS CLASSIFICATION

In [34]:
new_data_test['Attack'].value_counts()

Attack
0    395106
3     98022
5     79290
4      4871
2      2039
1       981
6       433
Name: count, dtype: int64

In [35]:
# 1. Count classes
class_counts_train = new_data_train['Attack'].value_counts()

# 2. Filter out small classes (where count < 900)
selected_classes_train = class_counts_train[class_counts_train >= 900] # Use >= to include 900
class_names_train = selected_classes_train.index

# 3. Filter the main DataFrame to include only selected classes
selected_train = new_data_train[new_data_train['Attack'].isin(class_names_train)] # FIXED variable name

dfs = []
for name in class_names_train:
    df = selected_train[selected_train['Attack'] == name]

    if len(df) > 2500:
        df = df.sample(n = 5000, random_state = 42, replace = False)

    dfs.append(df)

# 5. Combine all processed DataFrames (filtered small classes, subsampled large classes)
mc_train = pd.concat(dfs, ignore_index = True)

# 6. Print the distribution of the final DataFrame
print(mc_train['Attack'].value_counts()) # FIXED variable name

Attack
0    5000
4    5000
3    5000
5    5000
2    5000
6    1718
1     974
Name: count, dtype: int64


In [36]:

MAX_SAMPLES = 2500


class_counts_test = new_data_test['Attack'].value_counts()

# 2. Filter out small classes (where count < 900)
# We keep classes with 900 or more records.
selected_classes_test = class_counts_test[class_counts_test >= 400]
class_names_test = selected_classes_test.index

# 3. Filter the main DataFrame to include only selected classes
selected_test = new_data_test[new_data_test['Attack'].isin(class_names_test)] 

dfs_test = []

for name in class_names_test:
    df_test = selected_test[selected_test['Attack'] == name]

    if len(df_test) > MAX_SAMPLES:
        df_test = df_test.sample(n = MAX_SAMPLES, random_state = 42, replace = False)

    dfs_test.append(df_test)

# 5. Combine all processed DataFrames (filtered small classes, subsampled large classes)
mc_test = pd.concat(dfs_test, ignore_index = True)

# 6. Print the distribution of the final DataFrame
print("\nFinal Multiclass Test Data Distribution:")
print(mc_test['Attack'].value_counts())


Final Multiclass Test Data Distribution:
Attack
0    2500
3    2500
5    2500
4    2500
2    2039
1     981
6     433
Name: count, dtype: int64


In [37]:
mc_train.to_parquet(r'final\train_mc.parquet')
mc_test.to_parquet(r'final\test_mc.parquet')